# 第 4 章：資料清理、日期特徵與營運報表

本 Notebook 融合 `lesson04.ipynb` 與 `practice_ch04.py`，並修復原 Notebook 中的中文編碼亂碼。

**適合對象**：已具備 Pandas 選取、篩選與基本分組概念的學習者。

**學習目標**：
- 檢查缺失值與重複資料。
- 將文字日期轉成 `datetime`，並衍生年份、小時、星期與週末特徵。
- 篩選指定年份的工作階段與訂單。
- 建立每日營收、付款方式績效及每週營運報表。
- 使用完整性檢查確認彙總結果正確。

## 學習流程

1. 載入課程資料
2. 檢查缺失值與重複資料
3. 轉換日期型別
4. 建立時間特徵並篩選年份
5. 建立完成訂單分析表
6. 製作每日營收報表
7. 比較付款方式績效
8. 製作每週營運報表
9. 驗證彙總結果

## 1. 環境設定與資料載入

使用 `common.py` 的共用函式載入資料，可避免 Notebook 執行位置不同而找不到 CSV。`order_facts()` 則會建立只包含已完成訂單的營收分析表。

In [ ]:
import pandas as pd
from IPython.display import display

from common import ensure_packages, load_data, order_facts

ensure_packages()
data = load_data()

pd.set_option("display.max_columns", 30)
pd.set_option("display.float_format", "{:,.2f}".format)

customers = data["customers"]
orders = data["orders"]
order_items = data["order_items"]
sessions = data["sessions"]

print("資料載入完成。")

## 2. 檢查缺失值

`isna()` 會把缺失位置標記為 `True`，再用 `sum()` 計算各欄位的缺失筆數。依缺失筆數由大到小排列，可以優先檢查問題較多的欄位。

In [ ]:
session_missing = (
    sessions.isna()
    .sum()
    .sort_values(ascending=False)
    .rename("missing_count")
)

session_missing_report = session_missing.to_frame()
session_missing_report["missing_rate"] = (
    session_missing_report["missing_count"] / len(sessions)
)
display(session_missing_report)

**解讀提醒**：缺失值不一定代表錯誤。是否需要刪除或填補，必須根據欄位定義與業務情境判斷。

## 3. 檢查重複資料

本節進行兩種檢查：

- `sessions.duplicated()`：整列資料是否完全重複。
- `orders["order_id"].duplicated()`：訂單主鍵是否重複。

主鍵即使其他欄位不同，也不應重複。

In [ ]:
duplicate_summary = pd.DataFrame({
    "檢查項目": ["sessions 完全重複列", "orders.order_id 重複"],
    "重複筆數": [
        int(sessions.duplicated().sum()),
        int(orders["order_id"].duplicated().sum()),
    ],
})
duplicate_summary["結果"] = duplicate_summary["重複筆數"].eq(0).map(
    {True: "通過", False: "需檢查"}
)
display(duplicate_summary)

## 4. 日期型別轉換

CSV 中的日期通常先被讀成文字型別。使用 `pd.to_datetime()` 轉換後，才能可靠地取得年份、時段、星期或進行時間排序。

`errors="coerce"` 會把無法解析的值轉成 `NaT`，便於後續統計問題筆數。

In [ ]:
sessions_clean = sessions.copy()

print("轉換前型別：", sessions_clean["session_start"].dtype)
sessions_clean["session_start"] = pd.to_datetime(
    sessions_clean["session_start"], errors="coerce"
)
print("轉換後型別：", sessions_clean["session_start"].dtype)
print("無法解析的日期筆數：", sessions_clean["session_start"].isna().sum())

display(sessions_clean[["session_id", "session_start"]].head())

## 5. 從日期衍生時間特徵

Pandas 日期欄位的 `.dt` 存取器可取得不同時間資訊：

- `.dt.year`：年份。
- `.dt.hour`：小時，範圍 0～23。
- `.dt.day_name()`：英文星期名稱。
- `.dt.dayofweek`：星期一為 0、星期日為 6。

因此 `dayofweek >= 5` 代表星期六或星期日。

In [ ]:
sessions_clean["year"] = sessions_clean["session_start"].dt.year
sessions_clean["session_hour"] = sessions_clean["session_start"].dt.hour
sessions_clean["weekday"] = sessions_clean["session_start"].dt.day_name()
sessions_clean["is_weekend"] = (
    sessions_clean["session_start"].dt.dayofweek >= 5
)

display(
    sessions_clean[
        ["session_start", "year", "session_hour", "weekday", "is_weekend"]
    ].head()
)

## 6. 篩選 2025 年工作階段

日期轉換後，可直接使用 `.dt.year == 2025` 建立布林條件。使用 `.copy()` 明確建立獨立資料，避免後續修改產生 `SettingWithCopyWarning`。

In [ ]:
sessions_2025 = sessions_clean.loc[
    sessions_clean["session_start"].dt.year == 2025
].copy()

print(f"2025 年工作階段：{len(sessions_2025):,} 筆")
print(f"占全部工作階段：{len(sessions_2025) / len(sessions_clean):.2%}")

## 7. 轉換訂單日期

原課程的小練習要求轉換 `orders["order_date"]` 並衍生年份。為保留原始資料，先建立副本。

In [ ]:
orders_clean = orders.copy()
orders_clean["order_date"] = pd.to_datetime(
    orders_clean["order_date"], errors="coerce"
)
orders_clean["order_year"] = orders_clean["order_date"].dt.year

display(orders_clean[["order_id", "order_date", "order_year", "status"]].head())
print("訂單日期範圍：", orders_clean["order_date"].min().date(), "至", orders_clean["order_date"].max().date())

## 8. 建立完成訂單營收分析表

`order_facts(data)` 會執行三件事：

1. 計算每筆訂單品項營收。
2. 依 `order_id` 加總成訂單營收 `line_revenue`。
3. 合併訂單資料並只保留 `completed` 訂單。

後續每日、付款方式及每週報表都使用相同的 `facts`，確保統計口徑一致。

In [ ]:
facts = order_facts(data).copy()

print(f"完成訂單分析表：{len(facts):,} 筆")
print("order_date 型別：", facts["order_date"].dtype)
display(facts.head())

## 9. 每日營收報表

先把 `order_date` 取到「日期」層級，再依日期分組：

- `revenue`：每日訂單營收總和。
- `orders`：每日完成訂單數。
- `aov`：平均訂單金額，公式為營收 ÷ 訂單數。

`AOV` 是 Average Order Value 的縮寫。

In [ ]:
daily_revenue = (
    facts.assign(order_day=facts["order_date"].dt.date)
    .groupby("order_day", as_index=False)
    .agg(
        revenue=("line_revenue", "sum"),
        orders=("order_id", "count"),
    )
    .sort_values("order_day")
)
daily_revenue["aov"] = (
    daily_revenue["revenue"] / daily_revenue["orders"]
).round(2)

display(daily_revenue.head())

### 程式鏈的意思

- `.assign()`：建立暫時使用的日期欄位，不修改原本的 `facts`。
- `.groupby()`：將同一天的訂單放入同一組。
- `.agg()`：為不同輸出欄位指定來源欄位與統計方法。
- `.sort_values()`：將報表按日期由舊到新排列。

## 10. 付款方式績效

依付款方式分組，計算訂單數、總營收及平均訂單金額，再按總營收由高到低排序。

In [ ]:
payment_performance = (
    facts.groupby("payment_type", as_index=False)
    .agg(
        orders=("order_id", "count"),
        total_revenue=("line_revenue", "sum"),
        avg_order_value=("line_revenue", "mean"),
    )
    .sort_values("total_revenue", ascending=False)
)

display(payment_performance.round(2))

**解讀提醒**：總營收高可能來自訂單數較多，也可能來自平均訂單金額較高，因此三個指標應一起閱讀。

## 11. 每週營運報表

`.dt.to_period("W")` 將日期轉成週期間。相同週期間的訂單會被放在同一組，再計算每週營收、訂單數與 AOV。

In [ ]:
facts_with_week = facts.copy()
facts_with_week["order_week"] = (
    facts_with_week["order_date"].dt.to_period("W")
)

weekly_report = (
    facts_with_week.groupby("order_week", as_index=False)
    .agg(
        revenue=("line_revenue", "sum"),
        orders=("order_id", "count"),
    )
    .sort_values("order_week")
)
weekly_report["aov"] = (
    weekly_report["revenue"] / weekly_report["orders"]
).round(2)

display(weekly_report.head())

週期間通常顯示成例如 `2025-01-06/2025-01-12`，代表該週的開始日與結束日，而不是單一日期。

## 12. 完整性檢查

不論按日、付款方式或週彙總，各報表的訂單數加總都應等於 `facts` 的訂單總數；營收加總也應一致。

In [ ]:
source_orders = len(facts)
source_revenue = facts["line_revenue"].sum()

validation = pd.DataFrame({
    "報表": ["每日", "付款方式", "每週"],
    "訂單數加總": [
        daily_revenue["orders"].sum(),
        payment_performance["orders"].sum(),
        weekly_report["orders"].sum(),
    ],
    "營收加總": [
        daily_revenue["revenue"].sum(),
        payment_performance["total_revenue"].sum(),
        weekly_report["revenue"].sum(),
    ],
})
validation["訂單數一致"] = validation["訂單數加總"].eq(source_orders)
validation["營收一致"] = validation["營收加總"].sub(source_revenue).abs().lt(0.01)

display(validation)
print(f"來源訂單數：{source_orders:,}")
print(f"來源營收：{source_revenue:,.2f}")

## 13. 練習題

請建立「每月 × 付款方式」報表，包含：

- 完成訂單數 `orders`
- 總營收 `revenue`
- 平均訂單金額 `aov`

思考：不同月份的付款方式排名是否相同？

In [ ]:
# TODO：可先遮住以下參考答案，再自行完成。
monthly_payment = (
    facts.assign(order_month=facts["order_date"].dt.to_period("M"))
    .groupby(["order_month", "payment_type"], as_index=False)
    .agg(
        orders=("order_id", "count"),
        revenue=("line_revenue", "sum"),
    )
)
monthly_payment["aov"] = (
    monthly_payment["revenue"] / monthly_payment["orders"]
).round(2)

display(monthly_payment.head(12))

## 常見錯誤與延伸

**常見錯誤**：
- 日期仍是文字型別就使用 `.dt`，會發生錯誤。
- 看到缺失值便直接刪除，卻未確認其業務意義。
- 只檢查整列重複，忽略主鍵重複。
- 使用全部訂單製作營收報表，卻未先定義是否應排除取消或退款訂單。
- 只比較總營收，未同時查看訂單數與 AOV。

**延伸練習**：
- 加入每日或每週營收成長率。
- 比較平日與週末的工作階段數量。
- 將每週報表畫成營收折線圖。

## 重點整理

- 缺失值與重複資料是分析前的基本品質檢查。
- 日期轉成 `datetime` 後，才能使用 `.dt` 衍生時間特徵。
- `groupby()` 搭配命名聚合可建立清楚的營運報表。
- AOV 等於營收除以訂單數。
- 不同彙總層級的總數應回到相同來源數據，作為完整性驗證。